# Liminal geometry: diffusion time dependence

In this notebook, we explore how the diffusion time affects the measured
signal in a Monte Carlo simulation. We use a liminal geometry to
represent a population of cells, and we vary the diffusion time while
keeping the b-value and echo time fixed. This allows us to investigate
how the intracellular morphology of neurons, astrocytes, and microglia
influences the diffusion-weighted signal.

[`Liminal geometries`](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/liminal_geometry/)
are a convenient way to represent a statistical population of cells
without having to explicitly position each cell in a simulation box. In
this notebook, we will use liminal geometries to simulate diffusion
inside a single neuron morphology, as well as a mixed population of
neurons, astrocytes, and microglia.

Note that the extracellular space is assumed to be homogeneous in such
liminal geometries. If you want a more realistic representation of the
extracellular space, you will need to to use a fully packed geometry.
These are more computationally expensive to simulate and generate, but
they allow you to model the extracellular space in more detail,
including the effect of the cell packing configuration on diffusion.

Load the simulator, MRIBuilder, and the plotting backend:

In [1]:
using Random
using MCMRSimulator
using MRIBuilder
using CairoMakie

Random.seed!(1)
set_theme!(Theme(fontsize=20))

## 1. Create reusable diffusion-time sequences

For the next experiment, keep the b-value and echo time fixed while
varying the diffusion time. We use the short-pulse approximation: the
diffusion gradients are instantaneous (i.e.,
`gradient=(type=:instant, )`). This lets us specify the diffusion time
`Δ` and b-value directly, without worrying about the scanner
constraints. We also create a b=0 reference sequence at the same echo
time to normalize the measured signal.

In [1]:
diffusion_times = [5.0, 10.0, 20.0, 40.0, 60.0]  # ms
fixed_bval = 1.0
fixed_echo_time = 100.0  # ms
diffusion_orientation = [1.0, 0.0, 0.0]

diffusion_time_sequences = [
    MRIBuilder.DiffusionSpinEcho(
        bval=fixed_bval,
        TE=fixed_echo_time,
        Δ=diffusion_time,
        gradient=(type=:instant, orientation=diffusion_orientation,),
    )
    for diffusion_time in diffusion_times
]

b0_reference_sequence = MRIBuilder.DiffusionSpinEcho(
    bval=0.0,
    TE=fixed_echo_time,
    Δ=first(diffusion_times),
    gradient=(type=:instant, orientation=diffusion_orientation,),
)

full_sequences = [b0_reference_sequence, diffusion_time_sequences...]

MRIBuilder.plot_sequence(first(diffusion_time_sequences))


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************


<img width=600 height=450 style='object-fit: contain; height: auto;' src="data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAlgAAAHCCAYAAAAzc7dkAAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAgY0hSTQAAeiYAAICEAAD6AAAAgOgAAHUwAADqYAAAOpgAABdwnLpRPAAAAAlwSFlzAAAOxAAADsQBlSsOGwAAIABJREFUeAHtwQt4F3Sh8PHvtt8CBnGbMGDcFMZtQxSSi5CJgKCJGnHTkIsYaijkCROVEC0Fkcoi0YQipwwDzPSUYiCHwz1Dk9vxgYkioYHYBEwYMMb7/t/n7H327OGm/JxTvp9POPp/IUmSpGgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSWewhx56iLfeeosSbdq04dZbb0WSTkdAks5gf/jDH1izZg0lLr30Um699VYk6XQEJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAn4nNmzfzxhtvUNrVV1+NJEn68gvoMzFv3jwmTpxIaUVFRaSkpCBJkr7cApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoAl8QL7zwAvv376espKQkMjIyyMrKIiMjg7K2bNnC+vXrOZkuXbqQmZmJJEnS6Qp8QYwaNYp3332X40lKSmLw4MFMnz6d9PR0Sjz//PPcfvvtnMz8+fPp378/kiRJpyvwBZKamsq8efMoraioiHfeeYfHH3+cuXPncuDAAZ599lnKuummm+jduzfH07FjRyRJkmIIfIEkJydz9dVXcywjRowgKyuLP/7xj/zrX/8iPT2d0tq1a8fVV1+NJEnSZy3wJVG7dm2+9rWv8Ze//IUtW7bQpUsXPqn333+ft956i4YNG9KwYUNKe/vtt9m1axetW7emRo0aSJIkHU/gS+Sjjz4i4ayzzuLTSElJ4Vvf+hZpaWmsX7+eqlWrkvD+++/TsWNH6tWrx9q1a5EkSTqRwJfEkiVLeOWVV/j6179OVlYWZRUVFVFYWMixfOUrXyE5OZn09HQef/xxrrzySsaPH8/06dNJGD16NPv27WPx4sVUqlQJSZKkEwl8gRw+fJjBgwdT2pEjR9i+fTt/+9vf6NGjB3l5eRzLrbfeyq233sqxPPnkkwwZMoSEvn37MmLECB555BEGDBjArl27WLBgAQ8++CDt2rXjVKWmplK5cmWOHj3K4cOHSbjwwguRVLG88cYbJCcnU+KVV16hU6dOSKqY5syZQ/PmzanoAl8gxcXF/PGPf6S0o0ePcujQIRIyMjJIS0vjWLp27UqrVq04lubNm1Paww8/zMsvv8z111/Pvn37uOiiixg3bhyfxPjx4xk/fjybNm1i3LhxSKqYunbtiqQvjpSUFL4IAl8glSpVorCwkLJ2797NnXfeyW9+8xvq1KnDz3/+c8oaMmQIN910E6eievXq/Pa3v6Vnz56kpaWRm5tLcnIyn0Z2djYvvvgikiTpzBH4EqhTpw6/+tWvmDt3LosWLSKGHTt2kHDgwAHy8/Np0qQJkiRJpyLwJVG5cmUaNmzI7t27OV3vvPMOY8aMoWfPnmzevJmRI0eyYcMGqlevjiRJ0skEvmQ++ugjTkdxcTHDhg0jKSmJ2bNns2HDBi6//HJ+8IMfMHPmTCRJkk4m8CWSlpbGgQMH+PDDD6lVqxafxk9/+lP++7//m1mzZtGwYUMaNmzI0KFDmTVrFv3796d3795IkiSdSOBLpFWrVrz++uvcfffdzJgxg09qw4YN/OhHP6JXr16MHDmSEj//+c956aWXuOGGG9i4cSM1atRAkiTpeAJfEPPnz+fw4cOcyIMPPsiNN95I1apVKTFo0CC+9rWv0aJFC06mcuXKLFy4kJycHEqrXbs2K1asYMeOHRQWFlKjRg0kSZKOJ/AF0aVLF06mcePGNG7cmNIaNWpEo0aNOBVZWVlkZWVxLM2bN6d58+ZIkiSdTECSJElRBSTpS+Txxx9n9uzZlAghsHz5cmIoLCyke/fulHbrrbdy7bXXIkmlBSTpS2T79u2sWbOGEikpKcRy5MgR1qxZQ2n9+vVDksoKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoD+nwULFpCXl8emTZv46KOPOPvss2nTpg1jx44lJycHSZKkUxU4w3300UeMGjWKp59+moSGDRvSsmVL3n77bVatWsVvf/tbvv/97/PTn/4USZKkUxE4gx06dIgLL7yQjRs30rt3b2bOnEmjRo0osWzZMkaNGsXPfvYz6tWrx+23344kSdLJBM5gv/rVr9i4cSNDhw5l9uzZJCcnU9pFF13EokWLOPfcc7nvvvsYPXo0aWlpSJIknUjgDLVv3z7uv/9+qlWrxkMPPURycjLH0qhRI8aOHcvTTz/NqlWr6NmzJ5IkSScSOEO98sorFBQUcPPNN1O3bl1OZNKkSUyaNAlJkqRTEThDbd68mYTs7GwkSZJiCpyhtmzZQkKzZs0o66WXXuLRRx+

    MRIBuilder.plot_sequence(last(diffusion_time_sequences))

All five diffusion-weighted sequences have the same b-value (`1 ms/μm²`)
and echo time (`100 ms`); only the diffusion time changes, from `5` to
`60 ms`. Because the gradients are instantaneous, the chosen `Δ` values
can be shorter than the finite gradient pulses used in a
scanner-constrained acquisition. If we later change b-value, only the
gradient impulse strength needs to change; `Δ` and `TE` can remain
fixed.

## 2. Diffusion time dependence inside a neuron

We will first simulate the diffusion inside a single neuron morphology.
The easiest way to do this in MCMRSimulator is to use a liminal
geometry. Liminal geometry generally represents a statistical population
of cells, but if we set the extracellular fraction to zero, the
simulated volume is entirely intracellular. This lets us simulate
diffusion inside a single neuron morphology without having to explicitly
position it in a box. Load the neuron morphology from its SWC file, then
use it as the only cell template in a liminal geometry.

In [1]:
neuron = read_swc(joinpath(@__DIR__, "cell_morphologies", "neuron.swc"))
neuron_geometry = LiminalGeometry(
    [(1.0, neuron)],
    extracellular_fraction=0.0,
)

LiminalGeometry(Tuple{Float64, Any}[(1.0, 2344 finitecylinders(position = <unique values>, radius = <unique values>, connected_to = <unique values>, rotation = [1.0 0.0 0.0; 0.0 1.0 0.0; 0.0 0.0 1.0], R1_surface = 0.0, R1_inside = 0.0, R2_surface = 0.0, R2_inside = 0.0, off_resonance_surface = 0.0, off_resonance_inside = 0.0, use_spherical_endpoint = true, use_boundingbox = true, )
Not set: repeats, grid_resolution, dwell_time, density, permeability, relaxation and size_scale)], 0.0, 1.0)

There is also a sample “astrocyte.swc” or “microglia.swc” morphology in
the `cell_morphologies` folder that can be used instead of the neuron.

The example SWC cell morphologies were downloaded from NeuroMorpho.Org
(https://neuromorpho.org/) and are made available under CC BY 4.0
(https://creativecommons.org/licenses/by/4.0/). For reuse in
publications, please follow NeuroMorpho.Org’s Terms of Use
(https://neuromorpho.org/useterm.jsp), including its attribution and
citation requirements. Source information is retained in each SWC file’s
header.

Run the b=0 reference and the diffusion-time sequence set together.
Diffusivity and relaxation are held fixed; because all sequences have
the same echo time, the b=0 signal provides a common reference for the
signal attenuation.

In [1]:
neuron_simulation = Simulation(
    full_sequences;
    geometry=neuron_geometry,
    diffusivity=2.0,
    R1=1 / 1000,
    R2=1 / 80,
    verbose=false,
    timestep=1.,
)

neuron_signals = readout(
    neuron_simulation;
    target_snr=10,
    subset=Subset(inside=true),
)

6-element Vector{MCMRSimulator.Spins.SpinOrientationSum}:
 SpinOrientationSum(longitudinal=2.378569034550734, transverse=286.5047968593141, phase=89.99999375478956°, nspins=1000, snr=[1.3878930128640393e7, 1.3734657947069883e13, 2.1118387051770464e16])
 SpinOrientationSum(longitudinal=2.3785690345507864, transverse=122.36252938926994, phase=90.5878651274343°, nspins=1000, snr=[20.522030969553946, 21.757600669480574, 2.110777316548694e16])
 SpinOrientationSum(longitudinal=2.378569034553222, transverse=128.80288538837786, phase=92.73484320447014°, nspins=1000, snr=[20.96832086904852, 24.416937345507375, 2.1167273447567656e16])
 SpinOrientationSum(longitudinal=2.3785690345644657, transverse=149.29568541112044, phase=91.84196564966747°, nspins=1000, snr=[25.5460342022264, 29.449009584213112, 2.136749603644024e16])
 SpinOrientationSum(longitudinal=2.37856903461839, transverse=162.71855073105357, phase=90.54630775851385°, nspins=1000, snr=[28.454991858712233, 33.956765391683795, 2.1463398593

Compare the per-spin transverse signal for each diffusion time with the
b=0 reference. The changing attenuation reflects how the neuron’s
branched intracellular morphology restricts diffusion over different
time scales.

In [1]:
neuron_transverse = transverse.(neuron_signals; mean=true)
neuron_attenuation = neuron_transverse[2:end] ./ first(neuron_transverse)

neuron_figure = Figure(size=(700, 400))
neuron_axis = Axis(
    neuron_figure[1, 1];
    xlabel="Diffusion time (ms)",
    ylabel="Transverse signal attenuation (S/S₀)",
)
scatterlines!(neuron_axis, diffusion_times, neuron_attenuation, markersize=12)
neuron_figure

## 3. Mix neuron, astrocyte, and microglia morphologies

Now include all three SWC cell types in the liminal population. Load
each morphology separately, then pass it into the `LiminalGeometry`
constructor with its relative cell-number weight:

In [1]:
neuron = read_swc(joinpath(@__DIR__, "cell_morphologies", "neuron.swc"))
astrocyte = read_swc(joinpath(@__DIR__, "cell_morphologies", "astrocyte.swc"))
microglia = read_swc(joinpath(@__DIR__, "cell_morphologies", "microglia.swc"))

mixed_geometry = LiminalGeometry(
    geometries=[
        (1.0, neuron),
        (3.0, astrocyte),
        (4.0, microglia),
    ],
    extracellular_fraction=0.2,
)

LiminalGeometry(Tuple{Float64, Any}[(1.0, 2344 finitecylinders(position = <unique values>, radius = <unique values>, connected_to = <unique values>, rotation = [1.0 0.0 0.0; 0.0 1.0 0.0; 0.0 0.0 1.0], R1_surface = 0.0, R1_inside = 0.0, R2_surface = 0.0, R2_inside = 0.0, off_resonance_surface = 0.0, off_resonance_inside = 0.0, use_spherical_endpoint = true, use_boundingbox = true, )
Not set: repeats, grid_resolution, dwell_time, density, permeability, relaxation and size_scale), (3.0, 101 finitecylinders(position = <unique values>, radius = <unique values>, connected_to = <unique values>, rotation = [1.0 0.0 0.0; 0.0 1.0 0.0; 0.0 0.0 1.0], R1_surface = 0.0, R1_inside = 0.0, R2_surface = 0.0, R2_inside = 0.0, off_resonance_surface = 0.0, off_resonance_inside = 0.0, use_spherical_endpoint = true, use_boundingbox = true, )
Not set: repeats, grid_resolution, dwell_time, density, permeability, relaxation and size_scale), (4.0, 528 finitecylinders(position = <unique values>, radius = <unique 

The number weights sum to `10`, but `LiminalGeometry` normalizes them
internally to `0.1`, `0.3`, and `0.4`. They describe relative cell
numbers, not the volume fraction occupied by each cell type. So, even
though there are more microglia then neurons, the neuron might
contribute more due to its large volume. The separate
`extracellular_fraction=0.2` sets the extracellular volume to 20%,
leaving 80% intracellular. See the [MCMRSimulator liminal geometry
documentation](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/liminal_geometry/#liminal_geometry)
for more details.

Use the same sequence set and biophysical parameters to measure the
average signal across all spins in the mixed morphology:

In [1]:
mixed_simulation = Simulation(
    full_sequences;
    geometry=mixed_geometry,
    diffusivity=2.0,
    R1=1 / 1000,
    R2=1 / 80,
    verbose=false,
    timestep=1.0,
)

mixed_signals = readout(
    mixed_simulation;
    target_snr=10,
    max_spins=1000,
)

6-element Vector{MCMRSimulator.Spins.SpinOrientationSum}:
 SpinOrientationSum(longitudinal=2.378569034550747, transverse=286.5047968593215, phase=90.00000915621527°, nspins=1000, snr=[1.39440113443739e7, 1.3462886092939477e13, 2.3722775270061624e16])
 SpinOrientationSum(longitudinal=2.3785690345508, transverse=103.66920359062499, phase=90.02129447918759°, nspins=1000, snr=[17.277839644451106, 17.422140765670804, 2.371968234814426e16])
 SpinOrientationSum(longitudinal=2.3785690345532426, transverse=111.86067402785551, phase=92.06626465930546°, nspins=1000, snr=[18.499629600347145, 19.448905373756922, 2.3827500793510772e16])
 SpinOrientationSum(longitudinal=2.3785690345644914, transverse=134.75539078386933, phase=88.31608476102632°, nspins=1000, snr=[22.623651880699413, 25.23697691609411, 2.3897863617502388e16])
 SpinOrientationSum(longitudinal=2.3785690346183608, transverse=160.24007520347848, phase=88.24654749375277°, nspins=1000, snr=[27.55245829960568, 33.67633469314789, 2.4330219441

Like previously, we normalise the transverse signal to the b=0 reference
before plotting.

In [1]:
mixed_transverse = transverse.(mixed_signals; mean=true)
mixed_attenuation = mixed_transverse[2:end] ./ first(mixed_transverse)

mixed_figure = Figure(size=(700, 400))
mixed_axis = Axis(
    mixed_figure[1, 1];
    xlabel="Diffusion time (ms)",
    ylabel="Transverse signal attenuation (S/S₀)",
)
scatterlines!(mixed_axis, diffusion_times, mixed_attenuation, markersize=12)
mixed_figure

Feel free to change the relative contribution of the different cell
morphologies to see the effect on the results.

In more realistic simulations, you probably want to use a wider range of
cell morphologies to sample the many different cell shapes in the brain.
Cell morphologies can be downloaded from the
[NeuroMorpho.Org](https://neuromorpho.org/) database, as used here or
from single cell generators like the [Virtual Brain Cell
Generator](https://github.com/palombom/Virtual-Cell-Generator-v1.0) or
Octopus from (Brammerloh et al. 2026).

# 4. Adding permeability to the processes

In the simulation above there is no exchange between the intra- and
extracellular compartments. This means that we could have just
calculated the intracellular and extracellular signals separately and
then combined them with the appropriate weights. The liminal geometry
becomes more useful when we add permeability to the processes, which
allows spins to move between the intra- and extracellular compartments.

For each SWC-derived `FiniteCylinders` morphology, the first three
objects represent the soma and the remaining objects represent
processes. Permeability is a per-object surface property, so we assign a
vector with zero permeability for the soma and permeability `1e-3` for
each process. The value `1e-3` is related to the exchange rate between
compartments as described in section 2.5 of (Cottaar et al. 2026). We
modify copies of the morphologies so the impermeable baseline remains
available for comparison.

In [1]:
process_permeability = 1e-3
function with_permeable_processes(morphology)
    permeable_morphology = deepcopy(morphology)
    permeability = ones(length(permeable_morphology))
    permeability[1:3] .= 0.0
    permeability[4:end] .= process_permeability
    permeable_morphology.permeability = permeability
    permeable_morphology
end

permeable_neuron = with_permeable_processes(neuron)
permeable_astrocyte = with_permeable_processes(astrocyte)
permeable_microglia = with_permeable_processes(microglia)

permeable_mixed_geometry = LiminalGeometry(
    geometries=[
        (6.0, permeable_neuron),
        (3.0, permeable_astrocyte),
        (1.0, permeable_microglia),
    ],
    extracellular_fraction=0.2,
)

LiminalGeometry(Tuple{Float64, Any}[(6.0, 2344 finitecylinders(position = <unique values>, radius = <unique values>, connected_to = <unique values>, rotation = [1.0 0.0 0.0; 0.0 1.0 0.0; 0.0 0.0 1.0], R1_surface = 0.0, R1_inside = 0.0, R2_surface = 0.0, R2_inside = 0.0, off_resonance_surface = 0.0, off_resonance_inside = 0.0, use_spherical_endpoint = true, use_boundingbox = true, permeability = <unique values>, )
Not set: repeats, grid_resolution, dwell_time, density, relaxation and size_scale), (3.0, 101 finitecylinders(position = <unique values>, radius = <unique values>, connected_to = <unique values>, rotation = [1.0 0.0 0.0; 0.0 1.0 0.0; 0.0 0.0 1.0], R1_surface = 0.0, R1_inside = 0.0, R2_surface = 0.0, R2_inside = 0.0, off_resonance_surface = 0.0, off_resonance_inside = 0.0, use_spherical_endpoint = true, use_boundingbox = true, permeability = <unique values>, )
Not set: repeats, grid_resolution, dwell_time, density, relaxation and size_scale), (1.0, 528 finitecylinders(position 

Run the same acquisition and compare the combined mean signal with the
impermeable mixed-geometry result above:

In [1]:
permeable_simulation = Simulation(
    full_sequences;
    geometry=permeable_mixed_geometry,
    diffusivity=2.0,
    R1=1 / 1000,
    R2=1 / 80,
    verbose=false,
    timestep=1.0,
)

permeable_signals = readout(permeable_simulation; target_snr=10)
permeable_transverse = transverse.(permeable_signals; mean=true)
permeable_attenuation = permeable_transverse[2:end] ./ first(permeable_transverse)

permeability_figure = Figure(size=(700, 400))
permeability_axis = Axis(
    permeability_figure[1, 1];
    xlabel="Diffusion time (ms)",
    ylabel="Transverse signal attenuation (S/S₀)",
)
scatterlines!(permeability_axis, diffusion_times, mixed_attenuation, label="Impermeable", markersize=10)
scatterlines!(permeability_axis, diffusion_times, permeable_attenuation, label="Permeable processes", markersize=10)
axislegend(permeability_axis, position=:rb)
permeability_figure

Feel free to change the `process_permeability` value to see how it
affects the results. Note that if the permeability is too high, the
diffusion will become close to free diffusion, which means that the
simulator will have to generate a very large number of spins to produce
a reliable signal. This will make the simulation very slow, so it is
best to keep the permeability low for this workshop.

Brammerloh, Malte, Inès de Riedmatten, Juliette Beaubis, et al. 2026.
*OCTOPUS: A Versatile Open-Source Tool Creating Realistic Numerical
Brain Cells*. bioRxiv. <https://doi.org/10.64898/2026.09.02.748537>.

Cottaar, Michiel, Zhiyu Zheng, Karla Miller, Benjamin C. Tendler, and
Saad Jbabdi. 2026. “Multi-Modal Monte Carlo MRI Simulator of Tissue
Microstructure.” *Imaging Neuroscience* 4 (March): IMAG.a.1177.
<https://doi.org/10.1162/IMAG.a.1177>.